# 01 · Exploration and ABC / XYZ classification

**Business question.** A distributor with 10 stores and 50 products must decide how much and when to reorder.
Before forecasting anything we need to know *what demand looks like*: how it grows, when it peaks, and which
products deserve the most attention.

This notebook:
1. checks the data (input contract, validation report);
2. computes the descriptive KPIs **with SQL** in DuckDB;
3. looks at trend and weekly / yearly seasonality;
4. classifies items with **ABC** (value) and **XYZ** (variability).

All the heavy lifting lives in `src/`; this notebook only tells the story. Results come from
`python -m src.pipeline --all` (cached in `data/processed/work/results.joblib`).

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import joblib
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

from src.config import load_config

cfg = load_config(ROOT / "config" / "params.yaml")
RESULTS = ROOT / "data" / "processed" / "work" / "results.joblib"
FIG = ROOT / "reports" / "figures"
assert RESULTS.exists(), "Run `python -m src.pipeline --all` from the project root first (about 2 minutes)."
res = joblib.load(RESULTS)
pd.set_option("display.float_format", "{:,.3f}".format)
pd.set_option("display.max_columns", 30)
print(f"Data: {res.data.file_name} · {res.dm.n_series} series · test period {res.profile.test[0]} → {res.profile.test[1]}")

## 1 · The data

Any daily sales file goes through the same input adapter (`src/ingest.py`): columns are mapped, rows are summed
per day, missing days become zero sales, and the result is validated. The demo is the Kaggle
*Store Item Demand Forecasting Challenge*.

In [ ]:
display(Markdown(res.validation.to_markdown()))
res.data.sales.head()

In [ ]:
for note in res.profile.notes:
    print("•", note)

## 2 · Descriptive KPIs with SQL (DuckDB)

The standardised data is loaded into DuckDB (`sales`, `item_master`, `stores` and the view `sales_enriched`).
Every query lives in `sql/kpi_queries.sql`, one named block per KPI, and is checked against pandas in
`tests/test_sql.py`.

In [ ]:
from src.sql_runner import parse_queries

queries = parse_queries(ROOT / "sql" / "kpi_queries.sql")
print(queries["sales_by_year"])
res.sql["sales_by_year"]

In [ ]:
res.sql["kpi_overview"].T

## 3 · Trend and seasonality

In [ ]:
display(Image(FIG / "demand_trend.png"))
display(Image(FIG / "seasonality.png"))

In [ ]:
dow = res.sql["dow_index"].set_index("day_name")["dow_index"]
month = res.sql["month_index"].set_index("month_name")["month_index"]
years = res.sql["sales_by_year"].dropna(subset=["yoy_growth_pct"])
print(f"Busiest weekday: {dow.idxmax()} ({dow.max():.2f}) · quietest: {dow.idxmin()} ({dow.min():.2f})")
print(f"Peak month: {month.idxmax()} ({month.max():.2f}) · lowest: {month.idxmin()} ({month.min():.2f})")
print(f"Average yearly growth (units per day): {years['yoy_growth_pct'].mean():+.1f}%")

**Do all stores share the same weekly shape?** If they do, one global model can learn it for every series
(ADR-0002). We compare the day-of-week profile of each store.

In [ ]:
s = res.data.sales.assign(dow=res.data.sales["date"].dt.dayofweek)
prof = s.groupby(["store", "dow"])["sales"].mean().unstack()
prof = prof.div(prof.mean(axis=1), axis=0)
display(prof.round(3))
print(f"Largest gap between stores on the same weekday: {(prof.max() - prof.min()).max():.3f} index points")

## 4 · ABC / XYZ classification

* **ABC** ranks items by revenue in the validation period: A = first 80% of revenue, B = up to 95%, C = the rest.
  The demo has no prices, so a seeded synthetic price list is used (and ABC by volume is kept next to it).
* **XYZ** uses the coefficient of variation of *weekly* demand (weekly buckets remove the predictable weekday
  pattern): X < 0.5 ≤ Y < 1.0 ≤ Z.

In [ ]:
cls = res.classification
display(cls.head(10))
print(cls["abc"].value_counts().sort_index().to_dict(), cls["xyz"].value_counts().sort_index().to_dict())

In [ ]:
display(Image(FIG / "abc_pareto.png"))
display(Image(FIG / "abc_xyz_matrix.png"))

In [ ]:
agree = (cls["abc"] == cls["abc_volume"]).mean()
print(f"ABC by revenue and ABC by volume agree on {agree:.0%} of the items")
print(f"Weekly CV: min {cls['weekly_cv'].min():.2f}, median {cls['weekly_cv'].median():.2f}, max {cls['weekly_cv'].max():.2f}")

## 5 · Findings

In [ ]:
n_a = int((cls["abc"] == "A").sum())
share_a = cls.loc[cls["abc"] == "A", "revenue_share"].sum()
xyz = cls["xyz"].value_counts()
display(Markdown(f"""
* Demand grows by about **{years['yoy_growth_pct'].mean():+.1f}% per year** and has a strong **yearly season**
  (peak in **{month.idxmax()}**, index {month.max():.2f}; low in **{month.idxmin()}**, {month.min():.2f}).
* The **weekly pattern** is just as strong: {dow.idxmax()} is {dow.max() / dow.min() - 1:.0%} busier than
  {dow.idxmin()}, and every store has the same shape — one global model can learn it for all series.
* **{n_a} of {len(cls)} items** make {share_a:.0%} of the revenue (class A).
* Weekly demand is very stable: {int(xyz.get('X', 0))} items are **X**, {int(xyz.get('Y', 0))} Y and
  {int(xyz.get('Z', 0))} Z. Most of the day-to-day movement is *predictable* (trend + seasons), which is exactly
  what a forecast-driven reorder point can exploit and a static one cannot.
"""))